In [ ]:
import hashlib, json, shutil
from pathlib import Path
W = Path('/kaggle/working')
rep = {}
for cfg in sorted(Path('/kaggle/input').rglob('config.json')):
    d = cfg.parent
    if 'gemma-4-31b' not in str(d): continue
    key = 'w4a16' if 'w4a16' in str(d) else 'bf16_unquantized'
    t = d / 'chat_template.jinja'
    tc = json.loads((d / 'tokenizer_config.json').read_text()) if (d / 'tokenizer_config.json').exists() else {}
    rep[key] = {'dir': str(d), 'files': sorted(p.name for p in d.iterdir())[:40],
                'chat_template_jinja_sha256': hashlib.sha256(t.read_bytes()).hexdigest() if t.exists() else None,
                'template_in_tokenizer_config': bool(tc.get('chat_template'))}
    if t.exists(): shutil.copy(t, W / f'chat_template_{key}.jinja')
    gc = d / 'generation_config.json'
    if gc.exists(): shutil.copy(gc, W / f'generation_config_{key}.json')
rep['identical_templates'] = (rep.get('w4a16', {}).get('chat_template_jinja_sha256') == rep.get('bf16_unquantized', {}).get('chat_template_jinja_sha256'))
(W / 'template_check.json').write_text(json.dumps(rep, indent=1))
print(json.dumps(rep, indent=1))